In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/home/kavish31/mlops-credit-risk/mlops-credit-risk/data/raw/application_train.csv')

# Basic info
print("Shape:", df.shape)
print("\nTarget distribution:\n", df['TARGET'].value_counts(normalize=True))

# Check data types
print("\nData types:\n", df.dtypes.value_counts())

# Missing values summary (top 10 columns by % missing)
missing_pct = df.isnull().mean() * 100
print("\nTop 10 columns by missing %:\n", missing_pct.sort_values(ascending=False).head(10))

# First rows to see structure
print("\nFirst 2 rows (first 5 columns):\n", df.iloc[:2, :5])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# Assume df is already loaded
# Drop columns with >50% missing
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
df_clean = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns with >50% missing. Remaining columns: {df_clean.shape[1]}")

# Handle DAYS_EMPLOYED: impute 0, create missing flag
df_clean['DAYS_EMPLOYED_MISSING'] = df_clean['DAYS_EMPLOYED'].isnull().astype(int)
df_clean['DAYS_EMPLOYED'] = df_clean['DAYS_EMPLOYED'].fillna(0)

# Create debt-to-income ratio
df_clean['DEBT_TO_INCOME'] = df_clean['AMT_CREDIT'] / df_clean['AMT_INCOME_TOTAL']

# Separate features and target
X = df_clean.drop(columns=['TARGET', 'SK_ID_CURR'])  # SK_ID_CURR is just an ID
y = df_clean['TARGET']

# Identify numeric and categorical columns (simplified for demonstration)
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
# Keep only low-cardinality categoricals for one-hot encoding (e.g., <10 unique)
categorical_cols = [c for c in categorical_cols if X[c].nunique() < 10]

# Preprocessor: scale numeric, one-hot encode categoricals
preprocessor = ColumnTransformer([
    ('num', RobustScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")

Dropped 41 columns with >50% missing. Remaining columns: 81
Train size: (246008, 81), Test size: (61503, 81)
Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [ ]:
# Full pipeline with ColumnTransformer and model training
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

# Assume X_train, X_test, y_train, y_test are already defined from previous steps
# Identify numeric and categorical columns (excluding target and ID)
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
# Keep low-cardinality categoricals (e.g., <10 unique)
categorical_cols = [c for c in categorical_cols if X_train[c].nunique() < 10]

# Preprocessor
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ]), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

# Train and evaluate
results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipeline.fit(X_train, y_train)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    results[name] = auc
    print(f"{name}: ROC-AUC = {auc:.4f}")

print("\nBest model:", max(results, key=results.get))

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
import shap

# Train the best model (XGBoost) on the full preprocessed data
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'))
])
final_pipeline.fit(X_train, y_train)

# Get the preprocessed training data for SHAP (the output of the preprocessor)
X_train_processed = final_pipeline.named_steps['preprocessor'].transform(X_train)
# Convert to dense array if sparse (one-hot encoding may produce sparse)
if hasattr(X_train_processed, 'toarray'):
    X_train_processed = X_train_processed.toarray()

# Get feature names after preprocessing
# For numeric columns, names are original; for categorical, OneHotEncoder generates names
feature_names = (numeric_cols + 
                 list(final_pipeline.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .get_feature_names_out(categorical_cols)))

# Create SHAP explainer using the model's predict_proba (for binary classification)
explainer = shap.Explainer(final_pipeline.named_steps['classifier'], X_train_processed, feature_names=feature_names)

# Compute SHAP values for a subset of test data (e.g., 100 rows to keep fast)
X_test_processed = final_pipeline.named_steps['preprocessor'].transform(X_test)
if hasattr(X_test_processed, 'toarray'):
    X_test_processed = X_test_processed.toarray()
shap_values = explainer(X_test_processed[:100])  # SHAP values for class 1 (default)

# Global summary plot
shap.summary_plot(shap_values, X_test_processed[:100], feature_names=feature_names)

# Local force plot for two individual predictions (e.g., first two test samples)
shap.initjs()
for i in range(2):
    shap.force_plot(explainer.expected_value, shap_values[i].values, X_test_processed[i], feature_names=feature_names, matplotlib=True)

ModuleNotFoundError: No module named 'shap'